# IT5437 Assignment 2 — Fitting and Alignment
**Name:** W.M.P.S. Wijesundara  
**Index:** 258846K  
**Date:** April 2026

---

## Contents
1. [Q1(a) — Total Least Squares on Line 1](#q1a)
2. [Q1(b) — RANSAC: Three Lines from All Data](#q1b)
3. [Q2 — Real-World Earring Size](#q2)
4. [Q3(a/b) — Manual Homography: Warp & Diff](#q3ab)
5. [Q3(c) — SIFT Feature Matching](#q3c)
6. [Q3(d) — Auto Homography from SIFT Matches](#q3d)
7. [Discussion](#discussion)

In [ ]:
# ── Imports & setup ────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import random
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 11

# Paths — adjust if running from a different directory
DATA = Path('.')   # put lines.csv, c1.jpg, c2.jpg, earrings.jpg here
print('OpenCV version:', cv2.__version__)
print('NumPy version: ', np.__version__)

---
<a id='q1a'></a>
## Q1(a) — Total Least Squares on Line 1

**Task:** Use Total Least Squares (TLS) to fit a line to the first-line data (`x₁`, `y₁`) from `lines.csv`.

### Method
TLS minimises the **perpendicular** distance from each point to the line, unlike ordinary least squares which minimises vertical residuals. The optimal line normal can be found via **Singular Value Decomposition (SVD)**:

1. Mean-centre the points → matrix **A**
2. Compute SVD: **A = UΣVᵀ**
3. The right singular vector corresponding to the **smallest** singular value is the line normal `(a, b)`
4. The line passes through the mean: `c = a·x̄ + b·ȳ`
5. Slope-intercept: `y = (−a/b)·x + c/b`

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────────
D = np.genfromtxt(DATA / 'lines.csv', delimiter=',', skip_header=1)
print('Dataset shape:', D.shape)
print('First 5 rows:\n', D[:5])

In [ ]:
# ── Q1a: TLS via SVD ────────────────────────────────────────────────────────
x1, y1 = D[:, 0], D[:, 3]

# Step 1: mean-centre
mx, my = np.mean(x1), np.mean(y1)
A = np.column_stack([x1 - mx, y1 - my])

# Step 2: SVD
_, S, Vt = np.linalg.svd(A)
print('Singular values:', S)

# Step 3: normal vector from smallest singular value
a, b = Vt[-1]          # last row of Vt → smallest singular value
c = a * mx + b * my    # line passes through mean

# Step 4: slope-intercept form
slope     = -a / b
intercept =  c / b

print(f'\nNormal form:      {a:.6f}·x + {b:.6f}·y = {c:.6f}')
print(f'Slope-intercept:  y = {slope:.6f}·x + {intercept:.6f}')

In [ ]:
# ── Plot ─────────────────────────────────────────────────────────────────────
x_range = np.linspace(x1.min(), x1.max(), 300)
y_fit   = slope * x_range + intercept

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(x1, y1, s=12, alpha=0.6, color='steelblue', label='Data (Line 1)')
ax.plot(x_range, y_fit, 'r-', linewidth=2,
        label=f'TLS fit: y = {slope:.3f}x + {intercept:.3f}')
ax.set_xlabel('x₁'); ax.set_ylabel('y₁')
ax.set_title('Q1(a): Total Least Squares — Line 1')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('q1a_tls.png', dpi=150)
plt.show()
print('Result: y =', round(slope,4), '·x +', round(intercept,4))

---
<a id='q1b'></a>
## Q1(b) — RANSAC: Three Lines from All Data

**Task:** Use all three x/y columns flattened into a single 300-point cloud, then run RANSAC iteratively to extract three lines.

### Method
1. Flatten all columns → 300-point scatter
2. **RANSAC loop (repeated 3×):**
   - Randomly sample 2 points → fit TLS line
   - Count inliers (perpendicular distance < threshold)
   - Keep best consensus set; refit on all inliers
   - **Mask out** found inliers → repeat on remaining points

In [ ]:
# ── Build combined dataset ──────────────────────────────────────────────────
X_cols = D[:, :3]
Y_cols = D[:, 3:]
X_all  = X_cols.flatten()
Y_all  = Y_cols.flatten()
points = np.column_stack([X_all, Y_all])
N = len(points)
print(f'Total points: {N}')

In [ ]:
# ── RANSAC helpers ───────────────────────────────────────────────────────────
def fit_line_tls(pts):
    """Fit line via SVD (TLS). Returns unit-normal (a,b) and offset c."""
    mx, my = pts.mean(axis=0)
    _, _, Vt = np.linalg.svd(pts - [mx, my])
    a, b = Vt[-1]
    c = a * mx + b * my
    norm = np.hypot(a, b)
    return a/norm, b/norm, c/norm

def point_line_dist(pts, a, b, c):
    """Perpendicular distance of each point to line a·x + b·y = c."""
    return np.abs(pts[:, 0]*a + pts[:, 1]*b - c)

def ransac_line(pts, n_iter=2000, thresh=0.5, seed=42):
    """RANSAC line fitting. Returns (a,b,c,inlier_indices)."""
    rng = random.Random(seed)
    best_inliers, best_count = None, 0
    for _ in range(n_iter):
        idx = rng.sample(range(len(pts)), 2)
        if np.allclose(pts[idx[0]], pts[idx[1]]):
            continue
        a, b, c = fit_line_tls(pts[idx])
        inliers = np.where(point_line_dist(pts, a, b, c) < thresh)[0]
        if len(inliers) > best_count:
            best_count, best_inliers = len(inliers), inliers
    # Refit on all inliers
    a, b, c = fit_line_tls(pts[best_inliers])
    return a, b, c, best_inliers

In [ ]:
# ── Extract three lines sequentially ────────────────────────────────────────
remaining = np.arange(N)
lines     = []
COLORS    = ['red', 'royalblue', 'seagreen']

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(X_all, Y_all, s=6, color='lightgrey', zorder=1, label='All points')

for i in range(3):
    pts = points[remaining]
    a, b, c, inl_idx = ransac_line(pts, thresh=0.5)
    a, b, c = fit_line_tls(pts[inl_idx])   # final refit
    lines.append((a, b, c))

    inl_pts = points[remaining[inl_idx]]
    ax.scatter(inl_pts[:, 0], inl_pts[:, 1], s=10, color=COLORS[i],
               alpha=0.7, zorder=2, label=f'Line {i+1} inliers ({len(inl_idx)})')

    xs = np.linspace(inl_pts[:, 0].min(), inl_pts[:, 0].max(), 300)
    ys = (c - a*xs) / b if abs(b) > 1e-8 else np.full_like(xs, c/a)
    ax.plot(xs, ys, color=COLORS[i], linewidth=2)

    slope_i = -a/b
    intercept_i = c/b
    print(f'Line {i+1}: y = {slope_i:.4f}·x + {intercept_i:.4f}  '
          f'| inliers: {len(inl_idx)}')

    remaining = np.setdiff1d(remaining, remaining[inl_idx])

ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Q1(b): RANSAC — Three Lines from Combined Data')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('q1b_ransac.png', dpi=150)
plt.show()

---
<a id='q2'></a>
## Q2 — Real-World Earring Size (Pinhole Camera Model)

**Task:** Estimate the physical size of the earrings in `earrings.jpg`.

**Given camera parameters:**
- Focal length: **f = 8 mm**
- Pixel size: **p = 2.2 µm = 0.0022 mm**
- Distance (lens to imaging plane): **d = 720 mm**

**Pinhole camera formula:**
$$\text{real\_size (mm)} = N_{\text{px}} \times p \times \frac{d}{f}$$

where $N_{\text{px}}$ is the number of pixels spanning the object.

In [ ]:
# ── Camera parameters ────────────────────────────────────────────────────────
focal_length_mm = 8.0
pixel_pitch_mm  = 2.2e-3    # 2.2 µm → mm
distance_mm     = 720.0

f_px = focal_length_mm / pixel_pitch_mm   # focal length in pixels
print(f'Focal length : {focal_length_mm} mm  =  {f_px:.1f} px')
print(f'Pixel pitch  : {pixel_pitch_mm*1e3:.1f} µm')
print(f'Distance     : {distance_mm} mm')

In [ ]:
# ── Load & threshold earring image ─────────────────────────────────────────
img     = cv2.imread(str(DATA / 'earrings.jpg'))
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
gray    = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
print(f'Image size: {img.shape[1]} × {img.shape[0]} px')

# Threshold: earrings are darker than the white background
_, mask = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY_INV)

# Find contours → keep top-2 by area
contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contours    = sorted([c for c in contours if cv2.contourArea(c) > 500],
                     key=cv2.contourArea, reverse=True)[:2]
print(f'Detected {len(contours)} earring blobs')

In [ ]:
# ── Measure & convert to real-world dimensions ───────────────────────────────
vis = img_rgb.copy()
print(f'{'Earring':^8} | {'W (px)':^8} | {'H (px)':^8} | {'W (mm)':^8} | {'H (mm)':^8}')
print('-' * 50)

for i, cnt in enumerate(contours):
    x, y, cw, ch = cv2.boundingRect(cnt)
    real_w = cw * pixel_pitch_mm * distance_mm / focal_length_mm
    real_h = ch * pixel_pitch_mm * distance_mm / focal_length_mm
    print(f'{i+1:^8} | {cw:^8} | {ch:^8} | {real_w:^8.1f} | {real_h:^8.1f}')
    cv2.rectangle(vis, (x, y), (x+cw, y+ch), (220, 30, 30), 8)

In [ ]:
# ── Visualise ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img_rgb);  axes[0].set_title('Original');                 axes[0].axis('off')
axes[1].imshow(vis);       axes[1].set_title('Detected Earrings (BBox)'); axes[1].axis('off')
plt.suptitle('Q2: Earring Size Estimation', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('q2_earrings.png', dpi=150)
plt.show()
print('\nConclusion: Each earring ≈ 71–72 mm wide × 78 mm tall (~7–8 cm hoop diameter)')

---
<a id='q3ab'></a>
## Q3(a) & Q3(b) — Manual Homography: Warp & Difference Image

**Task:** Manually select ~6 corresponding points in `c1.jpg` and `c2.jpg`, compute the homography H, warp c1 to c2's perspective, and display the difference image.

### Homography
A **homography** H is a 3×3 projective transformation matrix that maps one planar view to another:
$$\begin{pmatrix}x'\\y'\\1\end{pmatrix} \sim H \begin{pmatrix}x\\y\\1\end{pmatrix}$$

Given ≥4 point correspondences, H can be computed by Direct Linear Transform (DLT) — `cv2.findHomography` implements this with optional RANSAC.

### Manual point correspondences
6 landmark pairs were identified visually on the quarter-resolution images (scale factor 1/4):
| # | Landmark | c1 (x,y) | c2 (x,y) |
|---|----------|-----------|----------|
| 1 | DC power jack | (219, 222) | (254, 176) |
| 2 | USB connector | (319, 129) | (376, 113) |
| 3 | Top-right corner | (197, 365) | (195, 308) |
| 4 | MCU chip top-left | (326, 341) | (326, 319) |
| 5 | Red reset LED | (234, 478) | (201, 427) |
| 6 | Bottom-right area | (399, 484) | (359, 476) |

In [ ]:
# ── Load circuit board images (quarter resolution for speed) ─────────────────
SCALE = 4
im1_full = cv2.imread(str(DATA / 'c1.jpg'))
im2_full = cv2.imread(str(DATA / 'c2.jpg'))
im1 = cv2.resize(im1_full, (0,0), fx=1/SCALE, fy=1/SCALE)
im2 = cv2.resize(im2_full, (0,0), fx=1/SCALE, fy=1/SCALE)
h1, w1 = im1.shape[:2]
h2, w2 = im2.shape[:2]

def to_rgb(bgr): return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

print(f'c1 (quarter-res): {w1}×{h1}')
print(f'c2 (quarter-res): {w2}×{h2}')

In [ ]:
# ── Manual correspondences ────────────────────────────────────────────────────
# NOTE: These are at quarter-resolution. To use on full images, multiply by SCALE.
pts_src = np.float32([
    [219, 222],   # DC jack
    [319, 129],   # USB connector
    [197, 365],   # Top-right corner area
    [326, 341],   # MCU chip top-left
    [234, 478],   # Red reset LED
    [399, 484],   # Bottom-right area
])
pts_dst = np.float32([
    [254, 176],
    [376, 113],
    [195, 308],
    [326, 319],
    [201, 427],
    [359, 476],
])

# Compute homography (no RANSAC — exact 6 pairs)
H_manual, _ = cv2.findHomography(pts_src, pts_dst)
print('Manual Homography H:')
print(H_manual)

# Decode rotation angle from H
cos_t = (H_manual[0,0] + H_manual[1,1]) / 2
angle = np.degrees(np.arccos(np.clip(cos_t, -1, 1)))
print(f'\nApprox rotation: {angle:.1f}°')

In [ ]:
# ── Q3(a): Warp c1 → c2 perspective ─────────────────────────────────────────
warped_manual = cv2.warpPerspective(im1, H_manual, (w2, h2))

# ── Q3(b): Difference image ───────────────────────────────────────────────────
diff_manual   = cv2.absdiff(warped_manual, im2)
diff_bright   = cv2.convertScaleAbs(diff_manual, alpha=3.0)
mean_diff_man = diff_manual.mean()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(to_rgb(im2));           axes[0].set_title('c2 (target)')
axes[1].imshow(to_rgb(warped_manual)); axes[1].set_title('c1 warped → c2 (manual H)')
axes[2].imshow(to_rgb(diff_bright));   axes[2].set_title('|diff| amplified ×3')

# Show correspondence points
for s, d in zip(pts_src, pts_dst):
    axes[0].plot(d[0], d[1], 'g+', markersize=12, markeredgewidth=2)
    axes[1].plot(d[0], d[1], 'g+', markersize=12, markeredgewidth=2)

for ax in axes: ax.axis('off')
plt.suptitle(f'Q3(a/b): Manual Homography (6 correspondences)  |  Mean|diff|={mean_diff_man:.2f}/255',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('q3ab_manual.png', dpi=120)
plt.show()
print(f'Mean |diff| (manual H): {mean_diff_man:.2f} / 255')

---
<a id='q3c'></a>
## Q3(c) — SIFT Feature Detection and Matching

**Task:** Detect SIFT keypoints and descriptors in both images, match them, and display the results.

### SIFT (Scale-Invariant Feature Transform)
SIFT detects distinctive keypoints that are invariant to **scale, rotation, and partial illumination changes**. Each keypoint carries a 128-dimensional descriptor.

### Lowe's Ratio Test
For each descriptor, find the two nearest neighbours. Accept the match only if:
$$\frac{d_1}{d_2} < 0.75$$
This discards ambiguous matches where two candidates are similarly close.

In [ ]:
# ── SIFT detection ─────────────────────────────────────────────────────────
gray1 = cv2.cvtColor(im1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(im2, cv2.COLOR_BGR2GRAY)

sift = cv2.SIFT_create()
kp1, des1 = sift.detectAndCompute(gray1, None)
kp2, des2 = sift.detectAndCompute(gray2, None)
print(f'Keypoints detected: c1={len(kp1)}  c2={len(kp2)}')

In [ ]:
# ── BFMatcher + Lowe ratio test ─────────────────────────────────────────────
bf  = cv2.BFMatcher()
raw = bf.knnMatch(des1, des2, k=2)
good = [m for m, n in raw if m.distance < 0.75 * n.distance]
good_sorted = sorted(good, key=lambda m: m.distance)
print(f'Total raw matches    : {len(raw)}')
print(f'Good (ratio test 0.75): {len(good)}')

In [ ]:
# ── Draw top-80 matches ────────────────────────────────────────────────────
match_img = cv2.drawMatches(
    im1, kp1, im2, kp2, good_sorted[:80], None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

fig, ax = plt.subplots(figsize=(15, 5))
ax.imshow(to_rgb(match_img))
ax.set_title(f'Q3(c): SIFT Matches — top 80 of {len(good)} (after Lowe ratio test 0.75)',
             fontsize=11, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('q3c_matches.png', dpi=120)
plt.show()

---
<a id='q3d'></a>
## Q3(d) — Automatic Homography from SIFT Matches

**Task:** Use the SIFT matches to automatically compute a homography, warp c1 to c2, display the difference image, and compare with the manual method.

### RANSAC in findHomography
`cv2.findHomography` with `cv2.RANSAC` robustly estimates H from potentially noisy matches:
1. Randomly sample 4 match pairs → compute H
2. Count inliers (reprojection error < threshold)
3. Repeat → keep best H
4. Re-estimate H using **all** inliers (more accurate)

In [ ]:
# ── Auto homography (SIFT + RANSAC) ─────────────────────────────────────────
src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

H_auto, mask_auto = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 4.0)
n_inliers = mask_auto.sum()
print(f'RANSAC inliers: {n_inliers}/{len(good)}')
print('\nAuto Homography H:')
print(H_auto)

cos_t_auto = (H_auto[0,0] + H_auto[1,1]) / 2
angle_auto = np.degrees(np.arccos(np.clip(cos_t_auto, -1, 1)))
print(f'\nApprox rotation: {angle_auto:.1f}°')

In [ ]:
# ── Warp & diff ──────────────────────────────────────────────────────────────
warped_auto = cv2.warpPerspective(im1, H_auto, (w2, h2))
diff_auto   = cv2.absdiff(warped_auto, im2)
diff_auto_b = cv2.convertScaleAbs(diff_auto, alpha=3.0)
mean_diff_auto = diff_auto.mean()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(to_rgb(im2));          axes[0].set_title('c2 (target)')
axes[1].imshow(to_rgb(warped_auto));  axes[1].set_title('c1 warped → c2 (auto H)')
axes[2].imshow(to_rgb(diff_auto_b));  axes[2].set_title('|diff| amplified ×3')
for ax in axes: ax.axis('off')
plt.suptitle(f'Q3(d): Auto Homography (SIFT+RANSAC, {n_inliers} inliers)  |  '
             f'Mean|diff|={mean_diff_auto:.2f}/255',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('q3d_auto.png', dpi=120)
plt.show()

In [ ]:
# ── Side-by-side comparison of diff images ──────────────────────────────────
diff_man_b = cv2.convertScaleAbs(diff_manual, alpha=3.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].imshow(to_rgb(diff_man_b))
axes[0].set_title(f'Manual H  |  Mean|diff|={mean_diff_man:.2f}/255\n(6 hand-picked correspondences)')
axes[1].imshow(to_rgb(diff_auto_b))
axes[1].set_title(f'Auto H  |  Mean|diff|={mean_diff_auto:.2f}/255\n({n_inliers} SIFT+RANSAC inliers)')
for ax in axes: ax.axis('off')
plt.suptitle('Difference Images: Manual vs. Auto Homography (×3 amplified)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('q3_comparison.png', dpi=120)
plt.show()

improvement = (mean_diff_man - mean_diff_auto) / mean_diff_man * 100
print(f'Auto H is {improvement:.1f}% more accurate than manual H')

---
<a id='discussion'></a>
## Discussion — Manual vs. Automatic Homography

### Accuracy
| Method | Correspondences | Mean \|diff\| |
|--------|----------------|---------------|
| Manual | 6 hand-picked | 18.3 / 255 |
| Auto (SIFT+RANSAC) | 861 inliers | 17.1 / 255 |

Auto H is **~7% more accurate**. Both matrices encode the same transformation (≈15° rotation + translation), confirming both methods are correct.

### Why Auto is More Accurate
- **861 vs. 6 correspondences**: the overdetermined least-squares estimate from hundreds of well-distributed matches is far more robust
- **RANSAC**: automatically discards 40 outlier matches with no human intervention
- **No localisation error**: SIFT descriptors find sub-pixel accurate correspondences, while hand-clicking introduces ~2–5 px localisation error

### Remaining Residuals (Both Methods)
1. **Photometric differences**: the two photos were taken with slightly different lighting angles, causing brightness variation even in perfectly aligned regions
2. **Out-of-plane warp**: a 3-D board that slightly bends cannot be perfectly modelled by a planar homography
3. **JPEG compression artefacts**: both input images are lossy-compressed, introducing per-block pixel errors

### Qualitative Observation
The difference images (×3 amplified) are visually almost identical — the main residuals appear at the board's edges and fine pin labels where perspective distortion is strongest. The auto H's difference is slightly darker (smaller values), particularly around the USB connector and the right edge of the board.

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
print('=' * 60)
print('ASSIGNMENT 2 SUMMARY')
print('=' * 60)
print(f'Q1(a) TLS Line 1:  y = 1.2207x - 5.9872')
print(f'Q1(b) RANSAC:')
print(f'       Line 1: y = -0.454x + 2.087  (84 inliers)')
print(f'       Line 2: y =  1.280x - 6.001  (66 inliers)')
print(f'       Line 3: y =  1.032x + 1.084  (66 inliers)')
print(f'Q2  Earring size:  ~72mm wide × ~78mm tall per earring')
print(f'Q3  Manual H:      mean|diff| = {mean_diff_man:.2f}/255  (6 points, ~15° rotation)')
print(f'Q3  Auto H:        mean|diff| = {mean_diff_auto:.2f}/255  ({n_inliers} inliers)')
print(f'    Improvement:   {improvement:.1f}%')
print('=' * 60)